<a href="https://colab.research.google.com/github/helbertlaverd-cell/Entregable-Helber-laverde/blob/master/Actividad_3_Entrega_Pedidos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller de Análisis Exploratorio de Datos (EDA)
## Cumplimiento de la promesa de entrega en Olist (e-commerce Brasil)

**Nombre:** _______________________________

**Dataset:** Brazilian E-Commerce Public Dataset by Olist — https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
**Archivo seleccionado:** `olist_orders_dataset.csv`

> **Nota de reproducibilidad.** Las cifras citadas en las interpretaciones se calculan en las celdas de código
> (celdas "Lectura automática"), de modo que el texto siempre coincide con lo que arrojan los datos al ejecutar el notebook.

# Fase 1. Comprensión del problema

## 1.1 Contexto
- **Tipo de organización:** Olist es un *marketplace* brasileño que conecta pequeños vendedores con grandes canales de venta. El vendedor despacha el producto y Olist/sus socios logísticos lo entregan al cliente. El dataset cubre ~100.000 pedidos entre 2016 y 2018.
- **Procesos estudiables:** ventas, pagos, logística, satisfacción (reseñas) y desempeño de vendedores.
- **Decisiones posibles:** fijar fechas de entrega prometidas, elegir socios logísticos, anticipar picos de demanda, detectar cuellos de botella operativos, reducir cancelaciones.
- **Información que falta:** costo del flete por pedido, causa de cancelación, capacidad logística, ubicación (en otras tablas), y cuántos clientes no compraron por una promesa de entrega larga.

## 1.2 Perspectiva profesional
Perspectiva de **operaciones y logística**: en comercio electrónico la entrega es el momento de verdad. Una entrega tardía deteriora la experiencia del cliente y la reputación del marketplace.

**Pregunta de negocio central:**
> ¿Qué tan bien cumple Olist la fecha de entrega que promete al cliente, y en qué etapa del proceso y en qué momentos se originan los incumplimientos?

**Indicadores a comprender:** tiempo de aprobación, tiempo de preparación del vendedor, tiempo de tránsito, tiempo total, holgura frente a la fecha estimada y porcentaje de pedidos entregados tarde.

In [ ]:
# Ideas iniciales / hipótesis de trabajo
ideas = [
    "La fecha estimada de entrega parece ser conservadora (holgura grande).",
    "Los retrasos se concentran en picos de demanda (p. ej. noviembre).",
    "El día/hora de compra afecta el tiempo de aprobación y de preparación.",
    "Los pedidos con preparación lenta del vendedor tienen mayor probabilidad de llegar tarde.",
]
for i, t in enumerate(ideas, 1):
    print(f"{i}. {t}")

# Fase 2. Selección y conocimiento de los datos

## 2.1 Selección de un único archivo

1. **Pregunta:** cumplimiento de la promesa de entrega y origen de los retrasos.
2. **Archivo necesario:** `olist_orders_dataset.csv`.
3. **Justificación:** es la única tabla que contiene **el ciclo completo de fechas del pedido** (compra → aprobación → entrega a transportadora → entrega al cliente → fecha estimada) y el **estado**. Con ella se calculan todos los indicadores de la pregunta sin necesidad de uniones. Se descartan: `order_items` (precio/flete, útil pero no responde la pregunta logística), `customers`/`sellers`/`geolocation` (ubicación, que enriquecería el análisis pero obliga a uniones), `payments` y `reviews` (otro foco: pago y satisfacción), `products` y `translation` (categorías).
4. **Una fila representa:** un pedido (`order_id`).
5. **Llaves para relacionar con otros archivos:** `order_id` (con items, pagos, reseñas) y `customer_id` (con clientes). *Ojo:* `customer_id` es único por pedido; el cliente real es `customer_unique_id`, que está en otra tabla.
6. **Riesgos al unir tablas:** un pedido tiene varios ítems y varios pagos, así que un `merge` con `order_items` duplica pedidos y puede inflar conteos y sumas.

**Archivo seleccionado:** `olist_orders_dataset.csv`

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 30)

In [ ]:
# Carga del archivo. Intenta, en orden: ruta local/Colab, kagglehub y subida manual (Colab).
NOMBRE = "olist_orders_dataset.csv"
rutas = [NOMBRE, f"/content/{NOMBRE}", f"data/{NOMBRE}", f"/kaggle/input/brazilian-ecommerce/{NOMBRE}"]
ruta = next((r for r in rutas if os.path.exists(r)), None)

if ruta is None:
    try:
        import kagglehub  # pip install kagglehub (requiere credenciales de Kaggle)
        carpeta = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
        ruta = os.path.join(carpeta, NOMBRE)
    except Exception as e:
        print("kagglehub no disponible:", type(e).__name__)

if ruta is None or not os.path.exists(ruta):
    try:
        from google.colab import files  # subida manual en Colab
        print("Sube el archivo olist_orders_dataset.csv")
        files.upload()
        ruta = NOMBRE
    except Exception:
        raise FileNotFoundError("Descarga olist_orders_dataset.csv de Kaggle y colócalo junto al notebook.")

orders_raw = pd.read_csv(ruta)
print("Archivo cargado desde:", ruta)

In [ ]:
# Objetos cargados y dimensiones
print("Objetos cargados: orders_raw (DataFrame)")
print("Dimensiones (filas, columnas):", orders_raw.shape)

## 2.2 Reconocimiento de la estructura

| Columna | Significado | Tipo esperado |
|---|---|---|
| `order_id` | Identificador del pedido | Identificador |
| `customer_id` | Identificador del cliente *en ese pedido* | Identificador |
| `order_status` | Estado actual del pedido | Categórica |
| `order_purchase_timestamp` | Momento de la compra | Fecha-hora |
| `order_approved_at` | Aprobación del pago | Fecha-hora |
| `order_delivered_carrier_date` | Entrega del vendedor a la transportadora | Fecha-hora |
| `order_delivered_customer_date` | Entrega al cliente | Fecha-hora |
| `order_estimated_delivery_date` | Fecha prometida al cliente | Fecha (sin hora útil) |

**Variables relevantes:** las cinco fechas y el estado. Los identificadores no se analizan estadísticamente.
**Variable poco evidente:** `order_estimated_delivery_date`; se investiga comparándola con la fecha de compra (holgura prometida).

In [ ]:
orders_raw.info()
print()
print(orders_raw.dtypes)

In [ ]:
display(orders_raw.head())
display(orders_raw.sample(5, random_state=42))
print("Valores únicos por columna:")
print(orders_raw.nunique())

## Apartado de apoyo: unión de tablas
No es necesaria para este EDA. Si se uniera `orders` con `order_items` por `order_id`, la relación sería **uno a muchos** (un pedido, varios ítems) y habría que verificar el número de filas antes y después. Aquí solo se verifica la unicidad de las llaves, que es el paso previo a cualquier unión.

In [ ]:
print("order_id único:   ", orders_raw["order_id"].is_unique)
print("customer_id único:", orders_raw["customer_id"].is_unique,
      "→ un customer_id por pedido; no sirve para medir recompra (se necesitaría customer_unique_id).")

# Fase 3. Calidad y confiabilidad de los datos

## 3.1 Diagnóstico

In [ ]:
# Valores faltantes
na = orders_raw.isna().sum().to_frame("faltantes")
na["porcentaje"] = na["faltantes"] / len(orders_raw) * 100
display(na[na["faltantes"] > 0].sort_values("faltantes", ascending=False))

# ¿Los faltantes se explican por el estado del pedido?
fechas = ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]
display(orders_raw.groupby("order_status")[fechas].apply(lambda g: g.isna().sum()).assign(
    total=orders_raw["order_status"].value_counts()))

In [ ]:
# Duplicados
print("Filas duplicadas exactas:", orders_raw.duplicated().sum())
print("order_id duplicados:     ", orders_raw["order_id"].duplicated().sum())

In [ ]:
# Tipos y categorías
print(orders_raw["order_status"].value_counts(dropna=False))
print("\nCategorías con espacios o mayúsculas inconsistentes:",
      (orders_raw["order_status"] != orders_raw["order_status"].str.strip().str.lower()).sum())

In [ ]:
# Conversión de fechas y verificación de coherencia cronológica
df = orders_raw.copy()
cols_fecha = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
              "order_delivered_customer_date", "order_estimated_delivery_date"]
for c in cols_fecha:
    df[c] = pd.to_datetime(df[c], errors="coerce")

p, a, c_, e, est = (df["order_purchase_timestamp"], df["order_approved_at"],
                    df["order_delivered_carrier_date"], df["order_delivered_customer_date"],
                    df["order_estimated_delivery_date"])

chequeos = {
    "Aprobado antes de la compra": (a < p).sum(),
    "Entregado a transportadora antes de la compra": (c_ < p).sum(),
    "Entregado a transportadora antes de aprobarse": (c_ < a).sum(),
    "Entregado al cliente antes de salir con la transportadora": (e < c_).sum(),
    "Entregado al cliente antes de la compra": (e < p).sum(),
    "Fecha estimada anterior a la compra": (est < p).sum(),
    "Estado 'delivered' sin fecha de entrega al cliente": ((df["order_status"] == "delivered") & e.isna()).sum(),
    "Fecha de entrega al cliente con estado distinto de 'delivered'": ((df["order_status"] != "delivered") & e.notna()).sum(),
}
display(pd.Series(chequeos, name="registros").to_frame())
print("Rango de compras:", p.min(), "→", p.max())

In [ ]:
# Valores sospechosos: tiempos extremos
tmp = pd.DataFrame({
    "total_dias": (e - p).dt.total_seconds() / 86400,
    "aprob_horas": (a - p).dt.total_seconds() / 3600,
})
display(tmp.describe(percentiles=[.5, .95, .99]).T)

# Cobertura mensual: ¿hay meses con muy pocos pedidos?
mensual = df.groupby(df["order_purchase_timestamp"].dt.to_period("M")).size()
display(mensual.head(6).to_frame("pedidos"))
display(mensual.tail(6).to_frame("pedidos"))

**Problema más importante.** Los faltantes en las fechas de aprobación, despacho y entrega **no son errores aleatorios**: están asociados al estado del pedido (un pedido cancelado o en tránsito aún no tiene fecha de entrega). Tratarlos como "datos perdidos" e imputarlos sería un error de interpretación. El segundo problema son los **pocos registros incoherentes** (fechas fuera de orden o entregas registradas en pedidos no entregados) y la **cobertura irregular en los extremos del periodo** (primeros meses de 2016 y últimos de 2018 con muy pocos pedidos).

## 3.2 Decisiones de tratamiento

| Problema | Evidencia | Alternativas | Decisión y justificación |
|---|---|---|---|
| Faltantes de fecha asociados al estado | Tabla de faltantes por estado | Imputar / eliminar / conservar | **Conservar.** Son información del proceso. Para métricas de entrega se usa el subconjunto `entregados` con fecha al cliente; para análisis de estados se usa la tabla completa. |
| Tipos de fecha como texto | `dtypes` = object | Mantener texto | **Convertir a `datetime`** para poder restar y agrupar. |
| Fechas fuera de orden cronológico | Tabla de chequeos | Corregir / eliminar / marcar | **Marcar con `flag_incoherente` y excluir solo de las métricas de duración.** No se corrige, porque no se sabe cuál fecha es la errada; se conserva el pedido para análisis de estado. |
| Cobertura irregular en extremos | Conteo mensual | Eliminar / conservar | **Excluir de las series temporales los meses con menos de 100 pedidos**, para no sacar conclusiones de muestras diminutas. |
| Duplicados | `duplicated()` | Eliminar | No hay evidencia de duplicados; no se elimina nada. |

In [ ]:
# Aplicación de las decisiones justificadas
df["flag_incoherente"] = (
    (a < p) | (c_ < p) | (c_ < a) | (e < c_) | (e < p) | (est < p)
).fillna(False)

# Variables derivadas (días)
dias = lambda x, y: (x - y).dt.total_seconds() / 86400
df["aprob_horas"] = (a - p).dt.total_seconds() / 3600
df["preparacion_dias"] = dias(c_, a)          # aprobación → entrega a transportadora
df["transito_dias"] = dias(e, c_)             # transportadora → cliente
df["total_dias"] = dias(e, p)                 # compra → cliente
df["promesa_dias"] = dias(est, p)             # compra → fecha prometida
df["holgura_dias"] = dias(est, e)             # >0: llegó antes de lo prometido
df["retraso_dias"] = -df["holgura_dias"]
df["tarde"] = df["holgura_dias"] < 0

df["anio_mes"] = df["order_purchase_timestamp"].dt.to_period("M")
df["dia_semana"] = df["order_purchase_timestamp"].dt.day_name()
df["hora"] = df["order_purchase_timestamp"].dt.hour

# Subconjunto para métricas de entrega
entregados = df[(df["order_status"] == "delivered")
                & df["order_delivered_customer_date"].notna()
                & ~df["flag_incoherente"]].copy()
meses_validos = mensual[mensual >= 100].index
print(f"Pedidos totales: {len(df):,} | Entregados analizables: {len(entregados):,} "
      f"({len(entregados)/len(df):.1%}) | Incoherentes marcados: {df['flag_incoherente'].sum():,}")

In [ ]:
# Verificación del efecto del tratamiento
print("Filas antes:", len(orders_raw), "| filas en df:", len(df), "(no se eliminó ningún pedido)")
print("Nuevas columnas:", [c for c in df.columns if c not in orders_raw.columns])
print("\nNulos en métricas derivadas (entregados):")
print(entregados[["aprob_horas", "preparacion_dias", "transito_dias", "total_dias", "promesa_dias", "holgura_dias"]].isna().sum())

# Fase 4. Comprensión estadística y análisis univariado

## 4.1 Variables numéricas
**Variables seleccionadas:** `aprob_horas`, `preparacion_dias`, `transito_dias`, `total_dias`, `promesa_dias` y `holgura_dias`. Son las que descomponen el recorrido del pedido y miden la promesa; el precio y el flete no están en este archivo.

In [ ]:
num = ["aprob_horas", "preparacion_dias", "transito_dias", "total_dias", "promesa_dias", "holgura_dias"]
res = entregados[num].describe(percentiles=[.25, .5, .75, .95, .99]).T
res["moda"] = [entregados[c].round(0).mode().iloc[0] for c in num]
res["asimetria"] = entregados[num].skew()
res["cv"] = res["std"] / res["mean"].abs()
display(res[["count", "mean", "50%", "moda", "std", "cv", "min", "95%", "99%", "max", "asimetria"]])

# Valores extremos por regla del IQR
for c in ["total_dias", "preparacion_dias", "transito_dias"]:
    q1, q3 = entregados[c].quantile([.25, .75]); iqr = q3 - q1
    n_out = (entregados[c] > q3 + 1.5 * iqr).sum()
    print(f"{c}: {n_out:,} atípicos altos ({n_out/len(entregados):.1%}) | umbral {q3 + 1.5*iqr:.1f} días")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, c in zip(axes.ravel(), num):
    s = entregados[c].dropna()
    sns.histplot(s.clip(upper=s.quantile(.99)), bins=50, ax=ax, color="#2a6f97")
    ax.axvline(s.mean(), color="crimson", ls="--", label=f"media {s.mean():.1f}")
    ax.axvline(s.median(), color="black", ls="-", label=f"mediana {s.median():.1f}")
    ax.set_title(c); ax.legend()
plt.suptitle("Distribuciones (recortadas al percentil 99 para visualizar)", y=1.02)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=entregados[["preparacion_dias", "transito_dias", "total_dias"]].clip(upper=60), orient="h", ax=ax, palette="Blues")
ax.set_title("Dispersión por etapa (días, recortado a 60)"); plt.show()

In [ ]:
# Lectura automática: variables numéricas
t, pr, tr = entregados["total_dias"], entregados["preparacion_dias"], entregados["transito_dias"]
print(f"• Tiempo total: media {t.mean():.1f} d, mediana {t.median():.1f} d, P95 {t.quantile(.95):.1f} d, máximo {t.max():.0f} d. "
      f"Media {'>' if t.mean()>t.median() else '≤'} mediana → asimetría positiva ({t.skew():.1f}): pocos pedidos muy demorados.")
print(f"• Preparación del vendedor: mediana {pr.median():.1f} d. Tránsito: mediana {tr.median():.1f} d. "
      f"El tránsito pesa {tr.median()/(pr.median()+tr.median()):.0%} del recorrido mediano post-aprobación.")
print(f"• Promesa al cliente: mediana {entregados['promesa_dias'].median():.1f} d vs entrega real mediana {t.median():.1f} d "
      f"→ holgura mediana {entregados['holgura_dias'].median():.1f} d.")

**Interpretación.** Las variables de tiempo son **asimétricas a la derecha**: la mediana describe mejor el caso típico y la media queda inflada por una cola larga de pedidos muy demorados. Los valores extremos **no se eliminan**: son pedidos reales (zonas remotas, problemas con el transportista) y justamente los que más afectan la experiencia del cliente. La visualización muestra la cola larga, la forma de la distribución y cuál etapa tiene más dispersión, cosas que un resumen de media y desviación no deja ver.

## 4.2 Variables categóricas

In [ ]:
estado = df["order_status"].value_counts().to_frame("pedidos")
estado["porcentaje"] = estado["pedidos"] / len(df) * 100
display(estado)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(x=estado.index, y=estado["pedidos"], ax=axes[0], palette="viridis")
axes[0].set_title("Pedidos por estado (escala lineal)"); axes[0].tick_params(axis="x", rotation=45)
sns.barplot(x=estado.index, y=estado["pedidos"], ax=axes[1], palette="viridis")
axes[1].set_yscale("log"); axes[1].set_title("Pedidos por estado (escala log: se aprecian los estados minoritarios)")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

orden_dias = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
df["dia_semana"].value_counts().reindex(orden_dias).plot.bar(ax=axes[0], color="#2a6f97"); axes[0].set_title("Pedidos por día de la semana")
df["hora"].value_counts().sort_index().plot.bar(ax=axes[1], color="#2a6f97"); axes[1].set_title("Pedidos por hora de compra")
plt.tight_layout(); plt.show()

In [ ]:
# Lectura automática: categóricas
ent = estado.loc["delivered", "porcentaje"] if "delivered" in estado.index else float("nan")
no_ent = estado.drop("delivered", errors="ignore")
print(f"• {ent:.1f}% de los pedidos figura como 'delivered'. El {100-ent:.1f}% restante se reparte en "
      f"{len(no_ent)} estados; los más frecuentes: " + ", ".join(f"{k} ({v:.1f}%)" for k, v in no_ent['porcentaje'].head(3).items()) + ".")
dia_top = df["dia_semana"].value_counts().idxmax(); hora_top = df["hora"].value_counts().idxmax()
print(f"• Día con más compras: {dia_top}. Hora pico: {hora_top}:00.")

**¿Qué descubrimiento obtuve?** La categoría dominante (`delivered`) es la más frecuente pero **no es la más informativa**: los estados minoritarios (cancelado, no disponible, en tránsito, facturado, procesando) concentran los problemas operativos y merecen investigación aparte aunque pesen poco en conteos. Además, aquí cada fila es un **pedido**: contar pedidos, clientes o unidades daría resultados distintos.

# Fase 5. Relaciones entre variables

## 5.1 Formulación de hipótesis

**Hipótesis 1 — Picos de demanda.** *Variables:* `anio_mes`, `tarde`, volumen de pedidos. *Espero:* mayor % de pedidos tardíos en meses de alto volumen (p. ej. noviembre, Black Friday), por saturación logística. *Se contradiría si:* el % tardío no sube en los meses de mayor volumen.

**Hipótesis 2 — Preparación lenta del vendedor.** *Variables:* `preparacion_dias`, `tarde`. *Espero:* los pedidos tardíos tienen un tiempo de preparación mayor. *Porque:* cada día extra antes de salir consume la holgura prometida. *Se contradiría si:* la preparación fuera igual en pedidos puntuales y tardíos (el retraso estaría en el tránsito).

**Hipótesis 3 — Momento de compra.** *Variables:* `dia_semana`/`hora`, `aprob_horas`, `total_dias`. *Espero:* las compras de fin de semana y de noche tardan más en aprobarse y en entregarse, porque los procesos de pago y despacho dependen de días hábiles. *Se contradiría si:* no hay diferencia de medianas entre grupos.

In [ ]:
# H1: picos de demanda
m = entregados.groupby("anio_mes").agg(pedidos=("order_id", "size"), pct_tarde=("tarde", "mean"),
                                        total_mediana=("total_dias", "median")).reset_index()
m = m[m["anio_mes"].isin(meses_validos)]
m["pct_tarde"] *= 100
rho, p1 = stats.spearmanr(m["pedidos"], m["pct_tarde"])
print(f"H1 | Spearman volumen mensual vs % tardío: rho={rho:.2f}, p={p1:.4f} (n={len(m)} meses)")

fig, ax1 = plt.subplots(figsize=(13, 5))
ax1.bar(m["anio_mes"].astype(str), m["pedidos"], color="#a9d6e5"); ax1.set_ylabel("Pedidos entregados")
ax1.tick_params(axis="x", rotation=90)
ax2 = ax1.twinx(); ax2.plot(m["anio_mes"].astype(str), m["pct_tarde"], color="crimson", marker="o"); ax2.set_ylabel("% entregados tarde")
ax2.grid(False); plt.title("H1 · Volumen mensual vs % de entregas tardías"); plt.tight_layout(); plt.show()

In [ ]:
# H2: preparación vs retraso
g = entregados.groupby("tarde")[["preparacion_dias", "transito_dias", "aprob_horas"]].median()
g.index = g.index.map({False: "A tiempo", True: "Tarde"})
display(g)
a_t = entregados.loc[~entregados["tarde"], "preparacion_dias"].dropna()
t_t = entregados.loc[entregados["tarde"], "preparacion_dias"].dropna()
u, p2 = stats.mannwhitneyu(t_t, a_t, alternative="greater")
print(f"H2 | Mann-Whitney (preparación tardíos > puntuales): p={p2:.2e}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, c in zip(axes, ["preparacion_dias", "transito_dias"]):
    sns.boxplot(data=entregados, x="tarde", y=entregados[c].clip(upper=40), ax=ax, palette=["#90be6d", "#f94144"])
    ax.set_xticklabels(["A tiempo", "Tarde"]); ax.set_title(c + " según cumplimiento (recortado a 40)")
plt.tight_layout(); plt.show()

corr = entregados[["aprob_horas", "preparacion_dias", "transito_dias", "total_dias", "promesa_dias", "retraso_dias"]].corr(method="spearman")
plt.figure(figsize=(7, 5)); sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0); plt.title("Correlación de Spearman"); plt.show()

In [ ]:
# H3: momento de compra
entregados["fin_de_semana"] = entregados["dia_semana"].isin(["Saturday", "Sunday"])
entregados["franja"] = pd.cut(entregados["hora"], [-1, 5, 11, 17, 23], labels=["Madrugada", "Mañana", "Tarde", "Noche"])

por_dia = entregados.groupby("dia_semana")[["aprob_horas", "total_dias"]].median().reindex(orden_dias)
por_franja = entregados.groupby("franja")[["aprob_horas", "total_dias"]].median()
display(por_dia); display(por_franja)

fds = entregados.loc[entregados["fin_de_semana"], "total_dias"].dropna()
lab = entregados.loc[~entregados["fin_de_semana"], "total_dias"].dropna()
u3, p3 = stats.mannwhitneyu(fds, lab)
h, pk = stats.kruskal(*[g["total_dias"].dropna() for _, g in entregados.groupby("dia_semana")])
print(f"H3 | Fin de semana vs laboral (total_dias): medianas {fds.median():.1f} vs {lab.median():.1f} d, p={p3:.2e}")
print(f"H3 | Kruskal-Wallis entre días: p={pk:.2e}")

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
por_dia["aprob_horas"].plot.bar(ax=axes[0], color="#2a6f97"); axes[0].set_title("Mediana de horas hasta aprobación, por día de compra")
por_dia["total_dias"].plot.bar(ax=axes[1], color="#577590"); axes[1].set_title("Mediana de días totales de entrega, por día de compra")
plt.tight_layout(); plt.show()

In [ ]:
# Lectura automática: hipótesis
dec = lambda p: "respaldada" if p < 0.05 else "no respaldada"
print(f"• H1 ({dec(p1 if rho>0 else 1)}): correlación volumen–% tardío rho={rho:.2f} (p={p1:.3f}). "
      f"Mes con mayor % tardío: {m.loc[m['pct_tarde'].idxmax(),'anio_mes']} ({m['pct_tarde'].max():.1f}%).")
print(f"• H2 ({dec(p2)}): mediana de preparación {t_t.median():.1f} d en tardíos vs {a_t.median():.1f} d en puntuales; "
      f"tránsito mediano {g.loc['Tarde','transito_dias']:.1f} vs {g.loc['A tiempo','transito_dias']:.1f} d.")
print(f"• H3 ({dec(p3)}): fin de semana {fds.median():.1f} d vs laboral {lab.median():.1f} d. "
      f"Efecto medido en días: {fds.median()-lab.median():+.1f}.")

**Cautela.** Estas pruebas indican asociación, no causalidad. Con ~100.000 filas casi cualquier diferencia resulta "estadísticamente significativa"; por eso se interpreta también el **tamaño del efecto en días**, que es lo que importa al negocio.

# Fase 6. Análisis temporal, segmentación o geográfico

## 6.1 Análisis temporal
**Pregunta:** ¿cómo evolucionó el volumen y el cumplimiento de la entrega entre 2016 y 2018, y existe estacionalidad?

*Limitaciones:* el dataset es una muestra de pedidos con reseña; los meses de 2016 y finales de 2018 tienen muy pocos registros, y solo cubre ~2 años, así que la estacionalidad anual no puede confirmarse con certeza.

In [ ]:
serie = df.groupby("anio_mes").size().rename("pedidos").to_frame()
serie = serie[serie.index.isin(meses_validos)]
t_m = entregados.groupby("anio_mes")["total_dias"].median().reindex(serie.index)
est_m = entregados.groupby("anio_mes")["promesa_dias"].median().reindex(serie.index)

x = serie.index.astype(str)
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(x, serie["pedidos"].values, marker="o", color="#2a6f97"); axes[0].set_title("Pedidos por mes")
axes[1].plot(x, t_m.values, marker="o", label="Entrega real (mediana)")
axes[1].plot(x, est_m.values, marker="s", ls="--", label="Promesa (mediana)")
axes[1].set_title("Tiempo real vs promesa al cliente (días)"); axes[1].legend(); axes[1].tick_params(axis="x", rotation=90)
plt.tight_layout(); plt.show()

# Heatmap día × hora
pivot = df.pivot_table(index="dia_semana", columns="hora", values="order_id", aggfunc="count").reindex(orden_dias)
plt.figure(figsize=(14, 4)); sns.heatmap(pivot, cmap="YlGnBu"); plt.title("Pedidos por día y hora de compra"); plt.show()

In [ ]:
# Lectura automática: temporal
pico = serie["pedidos"].idxmax()
print(f"• Mes de mayor volumen: {pico} ({serie['pedidos'].max():,} pedidos). Volumen del primer mes válido: "
      f"{serie['pedidos'].iloc[0]:,} → crecimiento de {serie['pedidos'].iloc[-1]/serie['pedidos'].iloc[0]:.1f}x hasta el último mes válido.")
print(f"• Mes con mayor mediana de entrega: {t_m.idxmax()} ({t_m.max():.1f} d); menor: {t_m.idxmin()} ({t_m.min():.1f} d).")

## 6.2 Segmentación
**Criterio:** segmentos por **desempeño de entrega**, según la holgura frente a la promesa: *Muy adelantado* (≥ 10 días antes), *Adelantado* (3–10 días), *Justo* (0–3 días), *Tarde* (hasta 7 días) y *Muy tarde* (> 7 días). Permite ver qué proporción del negocio opera con margen y qué proporción incumple y con cuánta severidad.

In [ ]:
bins = [-np.inf, -7, 0, 3, 10, np.inf]
labs = ["Muy tarde (>7 d)", "Tarde (0–7 d)", "Justo (0–3 d)", "Adelantado (3–10 d)", "Muy adelantado (≥10 d)"]
entregados["segmento"] = pd.cut(entregados["holgura_dias"], bins=bins, labels=labs)
seg = entregados.groupby("segmento").agg(pedidos=("order_id", "size"),
                                         prep_med=("preparacion_dias", "median"),
                                         trans_med=("transito_dias", "median"),
                                         promesa_med=("promesa_dias", "median"))
seg["%"] = seg["pedidos"] / seg["pedidos"].sum() * 100
display(seg)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
seg["%"].plot.barh(ax=axes[0], color=["#f94144", "#f3722c", "#f9c74f", "#90be6d", "#43aa8b"]); axes[0].set_title("% de pedidos por segmento")
seg[["prep_med", "trans_med"]].plot.barh(stacked=True, ax=axes[1], color=["#577590", "#f9844a"])
axes[1].set_title("Mediana de días por etapa en cada segmento")
plt.tight_layout(); plt.show()

In [ ]:
# Lectura automática: segmentación
tarde_pct = entregados["tarde"].mean() * 100
muy_tarde = (entregados["segmento"] == labs[0]).mean() * 100
print(f"• {tarde_pct:.1f}% de los pedidos entregados llegó después de la fecha prometida; {muy_tarde:.1f}% con más de 7 días de retraso.")
print(f"• El segmento más particular es '{labs[0]}': promesa mediana {seg.loc[labs[0],'promesa_med']:.0f} d, "
      f"preparación mediana {seg.loc[labs[0],'prep_med']:.1f} d y tránsito mediano {seg.loc[labs[0],'trans_med']:.1f} d.")

## 6.3 Análisis geográfico
**No aplica con el archivo elegido.** `olist_orders_dataset.csv` no contiene ubicación del cliente ni del vendedor (están en `customers` y `sellers`). Se descartó la unión para mantener el alcance de un único archivo.

*Qué no sería válido afirmar:* que ciertas regiones sufren más retrasos. La hipótesis geográfica (estados lejanos del centro de despacho tendrían tránsitos más largos) es el siguiente paso natural, uniendo `orders` con `customers` por `customer_id` (relación 1 a 1, sin riesgo de duplicación).

# Fase 7. Construcción de hallazgos
Cada hallazgo se imprime con las cifras calculadas sobre los datos, junto con su evidencia, interpretación y nivel de confianza.

In [ ]:
def n(x): return f"{x:,.0f}"
entregado_pct = (df["order_status"] == "delivered").mean() * 100
faltan_cliente = df["order_delivered_customer_date"].isna().sum()

hallazgos = [
 ("Holgura entre la promesa de entrega y la entrega real",
  f"Mediana de entrega real {t.median():.1f} d vs promesa {entregados['promesa_dias'].median():.1f} d (holgura mediana {entregados['holgura_dias'].median():.1f} d).",
  "`order_purchase_timestamp`, `order_estimated_delivery_date`, `order_delivered_customer_date`; diferencia de fechas y mediana.",
  "Una promesa muy conservadora protege el cumplimiento, pero puede reducir la conversión al mostrar plazos largos.",
  "Alta (cálculo directo)."),
 ("Magnitud y severidad de las entregas tardías",
  f"{tarde_pct:.1f}% de entregas tardías y {muy_tarde:.1f}% con más de 7 d de retraso; asimetría del total = {t.skew():.1f}.",
  "`holgura_dias`, `tarde`, segmentación por rangos.",
  "El promedio oculta los casos graves, que son los que más dañan la experiencia del cliente.",
  "Alta."),
 ("Peso relativo de cada etapa (preparación vs tránsito)",
  f"Mediana de tránsito {tr.median():.1f} d vs preparación {pr.median():.1f} d; CV tránsito {tr.std()/tr.mean():.2f} vs preparación {pr.std()/pr.mean():.2f}.",
  "`preparacion_dias`, `transito_dias`; estadísticas, boxplots.",
  "La mayor palanca de mejora está en la logística de última milla, y no solo en la velocidad del vendedor.",
  "Media-alta: sin ubicación no se sabe si es distancia o desempeño del transportista."),
 ("Etapa en la que se diferencian los pedidos tardíos",
  f"Preparación mediana {t_t.median():.1f} d (tardíos) vs {a_t.median():.1f} d (puntuales); tránsito {g.loc['Tarde','transito_dias']:.1f} vs {g.loc['A tiempo','transito_dias']:.1f} d.",
  "Comparación de medianas y prueba de Mann-Whitney (H2).",
  "Permite saber qué socio intervenir: vendedor (preparación) o transportista (tránsito).",
  "Media: es asociación, no causalidad."),
 ("Evolución mensual del volumen y del cumplimiento",
  f"Volumen último/primer mes válido: {serie['pedidos'].iloc[-1]/serie['pedidos'].iloc[0]:.1f}x; pico en {pico}; mes con mayor % tardío {m.loc[m['pct_tarde'].idxmax(),'anio_mes']} ({m['pct_tarde'].max():.1f}%); rho volumen–% tardío = {rho:.2f}.",
  "Serie mensual, Spearman (H1).",
  "Los picos de demanda exigen planificación de capacidad y una promesa dinámica.",
  "Media: solo ~2 años y una muestra con reseñas."),
 ("Efecto del momento de compra (día y hora) en el plazo de entrega",
  f"Fin de semana {fds.median():.1f} d vs laboral {lab.median():.1f} d (Δ {fds.median()-lab.median():+.1f} d); aprobación mediana de {entregados['aprob_horas'].median():.1f} h.",
  "`dia_semana`, `hora`, `aprob_horas`, `total_dias`; Mann-Whitney y Kruskal (H3).",
  "Se pueden ajustar promesas según día/hora de compra (p. ej. compras de viernes noche).",
  "Media-baja: la significancia estadística no implica relevancia práctica."),
 ("Casi todo el problema de calidad de datos es estructural, no ruido",
  f"{entregado_pct:.1f}% entregados; {n(faltan_cliente)} pedidos sin fecha de entrega al cliente, explicados por su estado; {n(df['flag_incoherente'].sum())} registros con fechas incoherentes.",
  "`isna()` por estado y chequeos de orden cronológico.",
  "Imputar o borrar nulos habría sesgado las métricas; hay que modelar el estado del pedido.",
  "Alta."),
]
for i, (h, ev, var, imp, conf) in enumerate(hallazgos, 1):
    print(f"Hallazgo {i}: {h}\n  Evidencia: {ev}\n  Variables/operaciones: {var}\n  Por qué importa: {imp}\n  Confianza: {conf}\n")

In [ ]:
# Visualización resumen para respaldar los hallazgos
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(entregados["holgura_dias"].clip(-30, 60), bins=60, ax=axes[0], color="#2a6f97")
axes[0].axvline(0, color="crimson", ls="--"); axes[0].set_title("Holgura frente a la promesa (días)\n<0 = tarde")
etapas = pd.DataFrame({"Preparación": [pr.median()], "Tránsito": [tr.median()]})
etapas.plot.barh(stacked=True, ax=axes[1], color=["#577590", "#f9844a"], legend=True); axes[1].set_title("Mediana de días por etapa")
sns.lineplot(data=m, x=m["anio_mes"].astype(str), y="pct_tarde", marker="o", ax=axes[2], color="crimson")
axes[2].tick_params(axis="x", rotation=90); axes[2].set_title("% de entregas tardías por mes")
plt.tight_layout(); plt.show()

# Fase 8. Conclusiones y recomendaciones

## 8.1 Conclusiones
> Las afirmaciones de esta sección deben contrastarse con las celdas de "Lectura automática" y la Fase 7; si las cifras ejecutadas difieren, ajusta la redacción.

**Pregunta inicial:** ¿qué tan bien cumple Olist su promesa de entrega y dónde se originan los incumplimientos?

**Conclusión 1.** Con un solo archivo (`orders`) se pudo reconstruir el recorrido del pedido en cuatro etapas y medir el cumplimiento. Se midió la holgura frente a la fecha prometida y la proporción de entregas tardías (Fases 4 y 6); como las distribuciones son asimétricas, el promedio oculta los casos graves y por eso se usaron medianas y percentiles.

**Conclusión 2.** Se comparó el aporte de cada etapa (hallazgos 3 y 4): el **tránsito** y la **preparación del vendedor** tienen palancas de mejora distintas, y la segunda es la que se puede intervenir directamente. Respaldamos las hipótesis según las pruebas de la Fase 5 (lectura automática de H1, H2 y H3); las que no resulten respaldadas deben reportarse como tales.

**Conclusión 3.** Los hallazgos son **asociaciones, no causas**, y el archivo carece de geografía, vendedor y transportista. El hallazgo más importante para el negocio es la **cola de pedidos muy tardíos**, que concentra el impacto en la experiencia del cliente.

## 8.2 Recomendaciones

**Recomendación 1 — Promesa dinámica de entrega** *(Hallazgos 1 y 5).* Calibrar la fecha estimada según periodo de alta demanda y momento de compra, en lugar de usar un margen uniforme. Se espera reducir los pedidos tardíos en picos sin alargar innecesariamente el plazo en meses normales.

**Recomendación 2 — Tablero de monitoreo por etapa** *(Hallazgos 3 y 4).* Seguir mensualmente la mediana y el P95 de preparación y tránsito, y el % de pedidos con retraso mayor a 7 días. Se espera detectar el cuello de botella (vendedor o transportista) antes de que se refleje en reseñas.

**Recomendación 3 — Intervención sobre la cola larga** *(Hallazgos 2 y 4).* Definir alertas operativas (p. ej. pedidos que superan la mediana de preparación en más de X días) para contactar al vendedor o reasignar transporte. Se espera disminuir los retrasos graves, que son los más costosos.

**Recomendación 4 — Siguiente análisis** *(Hallazgo 7 y 6.3).* Unir `orders` con `customers`, `sellers` y `reviews` para medir el efecto de la ubicación y del retraso sobre la calificación del cliente, y cuantificar así el costo real de entregar tarde.

## Limitaciones
Dataset de ~2 años, muestra de pedidos con reseña (posible sesgo de selección), sin costo logístico ni causas de cancelación, y análisis limitado a un archivo.